# Derive the triaxial ellipsoidal axes of halos and central galaxies from particle positions.
## Runtime: 20h

In [1]:
import numpy as np
import h5py
import multiprocessing
from stack_clusters_partaxis import process_halo_data
import os
import shutil
import pickle
import traceback

In [2]:
def task(snapnum, sgaldata_save_path, catalog_path, snapshot_path, part_axis, L, simulation_name, h):
    N_re = 50
    
    dir_path0 = sgaldata_save_path + 'GalInHalo_' + part_axis + f'_axis/{snapnum:03}'

    if not os.path.exists(dir_path0):
        os.makedirs(dir_path0)
    
    sgaldata, CID_vec = process_halo_data(simulation_name, h, catalog_path, snapshot_path, L, snapnum, star_num_limit=100, halo_mass_limit=1e12, include_cgal=True, part_axis='star', rho_N_cri_ratio=3, N_re=N_re)
    # sgaldata: (x, y, z, r, RA, DEC sfr, starmass, CatalogueID, central_galaxy_CatalogueID, progen_CatalogueID, 
    #            progen_central_galaxy_CatalogueID, descen_CatalogueID, descen_central_galaxy_CatalogueID , 
    #            vx, vy, vz, R200, Rgal_half, sgal_abspos(x, y, z), cgal_abspos(x, y, z), sgal_absvel(x, y, z), 
    #            cgal_absvel(x, y, z), gal_total_mass, Halo_M200c, whether_merge, gal_dmmass, gal_gasmass)
    with open(sgaldata_save_path + 'GalInHalo_' + part_axis + f'_axis/{snapnum:03}/CID_vec.pkl', 'wb') as f:
        pickle.dump(CID_vec, f)
    for i in range(len(sgaldata)):
        if (i == 0):
            sgaldata_arr_i = np.hstack(sgaldata[i]).T
            np.save(sgaldata_save_path + 'GalInHalo_' + part_axis + f'_axis/{snapnum:03}/galaxy_' + f'ave' + '.npy', sgaldata_arr_i)
        if (i != 0):
            sgaldata_arr_i = np.hstack(sgaldata[i]).T
            np.save(sgaldata_save_path + 'GalInHalo_' + part_axis + f'_axis/{snapnum:03}/galaxy_' + f'{i-1}' + '.npy', sgaldata_arr_i)
    print(part_axis + f", Progress: {snapnum}")

In [ ]:
with open("classifier_log.txt", "w", buffering=1) as f:

    part_axis_list = ['star', 'dm']
    simulations = [
        {
            "name": "SIMBA",
            "snapnum": "151",
            "sgaldata_save_path": '/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/SIMBAm100n1024/data/',
            "catalog_path": '/data/inspur_disk06/userdir/zhangzhuoming/Simba/galaxy_catalogs/m100n1024_151.hdf5',
            "snapshot_path": '/data/inspur_disk06/userdir/zhangzhuoming/Simba/snapshot/snap_m100n1024_151.hdf5',
            'h': 0.68, 
            "L": 1e5 / 0.68 # box size [kpc]
        },
        {
            "name": "TNG100",
            "snapnum": "099",
            "sgaldata_save_path": '/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/TNG100/data/',
            "catalog_path": '/data/inspur_disk01/userdir/zhangzhuoming/TNG100/galaxy_catalogs/Caesar_TNG_100.hdf5',
            "snapshot_path": "/data/inspur_disk02/Simulations/Illustris/TNG100-1/output/snapdir_099/snap_099.0.hdf5",
            'h': 0.6774, 
            "L": 75000.0 / 0.6774  # box size [kpc]
        },
        {
            "name": "EAGLE",
            "snapnum": "028_z000p000",
            "sgaldata_save_path": '/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/EAGLE/data/',
            "catalog_path": '/data/inspur_disk01/userdir/zhangzhuoming/EAGLE/galaxy_catalogs/Caesar_RefL0100N1504-FoF.hdf5',
            "snapshot_path": '/data/inspur_disk02/Simulations/Eagle/RefL01001504/snapshot_028_z000p000/snap_028_z000p000.0.hdf5',
            'h': 0.6777, 
            "L": 1e5 # box size [kpc]
        },
        {
            "name": "SIMBAnofb",
            "snapnum": "151",
            "sgaldata_save_path": '/data/inspur_disk06/userdir/zhangzhuoming/Quench_Direction_Data/SIMBAm50n512/nofb/data/',
            "catalog_path": '/data/inspur_disk06/userdir/zhangzhuoming/Simba/galaxy_catalogs_nofb/m50n512_151.hdf5',
            "snapshot_path": '/data/inspur_disk06/userdir/zhangzhuoming/Simba/snapshot_nofb/snap_m50n512_151.hdf5',
            'h': 0.68, 
            "L": 5e4 / 0.68  # box size [kpc]
        }
    ]


    for sim in simulations:
        print(f"=== Processing Simulation: {sim['name']} ===", file=f)

        for part_axis in part_axis_list:
            print(f"Running task for part_axis: {part_axis}", file=f)

            try:
                
                task( 
                    snapnum=sim["snapnum"],
                    sgaldata_save_path=sim["sgaldata_save_path"],
                    catalog_path=sim["catalog_path"],
                    snapshot_path=sim["snapshot_path"],
                    part_axis=part_axis,
                    L=sim["L"],
                    simulation_name=sim["name"],
                    h=sim['h'],
                )
                
                print(f"Finish: {sim['name']}-{part_axis}", file=f)
            except Exception:
                # Catch Error message
                traceback.print_exc(file=f)

Loading particle index: 100%|██████████| 512/512 [00:00<00:00, 1815.42it/s]
